In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score, mean_squared_error

sys.path.append(str(Path.cwd() / "src"))
from logistic import LogisticRegressionScratch
from tree import DecisionTreeScratch
from boosting import GradientBoostingScratch
from scipy.special import expit

EPS = 1e-3   # probabilities are clipped to [EPS, 1-EPS] for every model, so log-loss is comparable


def boost_early_stop(task, X_tr, y_tr, max_trees=150, lr=0.1, depth=3, seed=0):
    """Pick the number of trees on an inner validation split, then refit on all training data."""
    strat = y_tr if task == "classification" else None
    X_a, X_v, y_a, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=seed, stratify=strat)
    m = GradientBoostingScratch(task, n_estimators=max_trees, learning_rate=lr, max_depth=depth).fit(X_a, y_a)
    losses = []
    for F in m.staged_decision_function(X_v):
        if task == "classification":
            losses.append(log_loss(y_v, np.clip(expit(F), EPS, 1 - EPS), labels=[0, 1]))
        else:
            losses.append(mean_squared_error(y_v, F))
    best = int(np.argmin(losses)) + 1
    final = GradientBoostingScratch(task, n_estimators=best, learning_rate=lr, max_depth=depth).fit(X_tr, y_tr)
    return final, best


def clf_scores(p, y_te):
    p = np.clip(p, EPS, 1 - EPS)
    return {"log-loss": log_loss(y_te, p, labels=[0, 1]),
            "accuracy": accuracy_score(y_te, p > 0.5),
            "AUC": roc_auc_score(y_te, p) if len(set(p)) > 1 else 0.5}


# ============ Classification: breast cancer ============
X, y = load_breast_cancer(return_X_y=True)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
rows, best_trees, t0 = [], [], time.time()
for fold, (tr, te) in enumerate(cv.split(X, y)):
    X_tr, X_te, y_tr, y_te = X[tr], X[te], y[tr], y[te]
    sc = StandardScaler().fit(X_tr)
    Xs_tr, Xs_te = sc.transform(X_tr), sc.transform(X_te)
    out = {}
    out["Prior (majority class)"] = np.full(len(te), y_tr.mean())
    out["Logistic regression (mine)"] = LogisticRegressionScratch(lam=1.0 / len(tr)).fit(Xs_tr, y_tr).predict_proba(Xs_te)
    out["Decision tree, depth 3 (mine)"] = DecisionTreeScratch("classification", max_depth=3).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    gb, best = boost_early_stop("classification", X_tr, y_tr, seed=fold)
    best_trees.append(best)
    out["Gradient boosting (mine, early stopping)"] = gb.predict_proba(X_te)
    out["Gradient boosting (sklearn, 100 trees)"] = GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    for name, p in out.items():
        rows.append({"fold": fold, "model": name, **clf_scores(p, y_te)})
    print(f"classification fold {fold + 1}/15 done ({time.time() - t0:.0f}s)")
clf_res = pd.DataFrame(rows)

# ============ Regression: diabetes ============
Xd, yd = load_diabetes(return_X_y=True)
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=0)
rows, best_trees_r, t0 = [], [], time.time()
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
for fold, (tr, te) in enumerate(cv.split(Xd)):
    X_tr, X_te, y_tr, y_te = Xd[tr], Xd[te], yd[tr], yd[te]
    out = {}
    out["Mean baseline"] = np.full(len(te), y_tr.mean())
    out["Linear regression (sklearn)"] = LinearRegression().fit(X_tr, y_tr).predict(X_te)
    out["Decision tree, depth 3 (mine)"] = DecisionTreeScratch("regression", max_depth=3, min_samples_leaf=5).fit(X_tr, y_tr).predict(X_te)
    gb, best = boost_early_stop("regression", X_tr, y_tr, seed=fold)
    best_trees_r.append(best)
    out["Gradient boosting (mine, early stopping)"] = gb.predict(X_te)
    out["Gradient boosting (sklearn, 100 trees)"] = GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr).predict(X_te)
    for name, p in out.items():
        rows.append({"fold": fold, "model": name, "RMSE": rmse(y_te, p)})
    print(f"regression fold {fold + 1}/15 done ({time.time() - t0:.0f}s)")
reg_res = pd.DataFrame(rows)


# ============ Summary tables ============
def summarise(df, metrics):
    g = df.groupby("model", sort=False)[metrics]
    return pd.concat([g.mean().round(4).add_suffix(" (mean)"), g.std().round(4).add_suffix(" (sd)")], axis=1)

print("\nCLASSIFICATION (breast cancer), 5-fold x 3 repeats")
display(summarise(clf_res, ["log-loss", "accuracy", "AUC"]))
print("trees chosen by early stopping: min", min(best_trees), "| median", int(np.median(best_trees)), "| max", max(best_trees))

print("\nREGRESSION (diabetes), 5-fold x 3 repeats")
display(summarise(reg_res, ["RMSE"]))
print("trees chosen by early stopping: min", min(best_trees_r), "| median", int(np.median(best_trees_r)), "| max", max(best_trees_r))


# ============ Paired differences with a corrected resampled t-test (Nadeau & Bengio) ============
def paired(df, a, b, metric, n_train, n_test):
    d = (df[df.model == a].sort_values("fold")[metric].values
         - df[df.model == b].sort_values("fold")[metric].values)
    k = len(d)
    se = np.sqrt((1.0 / k + n_test / n_train) * d.var(ddof=1))
    half = stats.t.ppf(0.975, k - 1) * se
    return {"A": a, "B": b, "metric": metric, "mean(A - B)": d.mean(),
            "95% CI low": d.mean() - half, "95% CI high": d.mean() + half,
            "CI excludes 0": bool(d.mean() - half > 0 or d.mean() + half < 0)}

B = "Gradient boosting (mine, early stopping)"
rowsC = [paired(clf_res, B, "Logistic regression (mine)", m, 455, 114) for m in ["log-loss", "AUC"]]
rowsC += [paired(clf_res, B, "Prior (majority class)", "log-loss", 455, 114),
          paired(clf_res, "Logistic regression (mine)", "Prior (majority class)", "log-loss", 455, 114),
          paired(clf_res, B, "Gradient boosting (sklearn, 100 trees)", "log-loss", 455, 114)]
rowsR = [paired(reg_res, B, "Linear regression (sklearn)", "RMSE", 353, 89),
         paired(reg_res, B, "Mean baseline", "RMSE", 353, 89),
         paired(reg_res, "Linear regression (sklearn)", "Mean baseline", "RMSE", 353, 89),
         paired(reg_res, B, "Gradient boosting (sklearn, 100 trees)", "RMSE", 353, 89)]
print("\nPaired differences (A - B). For log-loss and RMSE, negative means A is better; for AUC, positive means A is better.")
display(pd.DataFrame(rowsC + rowsR).round(4))

Path("reports").mkdir(exist_ok=True)
clf_res.to_csv("reports/clf_cv_results.csv", index=False)
reg_res.to_csv("reports/reg_cv_results.csv", index=False)
print("Saved per-fold results in reports/")